### Input Data

This notebook uses two preprocessed input files:
- `stations_coordinates.csv`: GNSS station metadata (coordinates and IDs)
- `merged_gnss_weather.csv`: Daily GNSS displacements merged with ERA5 environmental variables

These files are derived from publicly available raw datasets (EarthScope GNSS and Copernicus ERA5) and are provided to ensure reproducibility of the modeling and explainability workflow.


# Explainable GAT-LSTM for GNSS Displacement Modeling

## 1. Data Loading and Preprocessing

## 2. GAT-LSTM Model Definition

## 3. Model Training

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.spatial import distance_matrix
import torch.nn.functional as F
# ======================
# 1. Load data
# ======================
stations = pd.read_csv("stations_coordinates.csv")
data = pd.read_csv("merged_gnss_weather.csv")

data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(['station_id','date'])

features = ["dN","dE","dU","skt","sp","src","ssr","stl1","swvl1","tp"]
targets  = ["dN","dE","dU"]

# Map station_id → index
station_map = {sid: i for i, sid in enumerate(stations['station_id'])}
data['station_idx'] = data['station_id'].map(station_map)

# ======================
# 2. Station-wise normalization
# ======================
scalers = {}
for sid, df in data.groupby("station_id"):
    scaler = MinMaxScaler()
    idx = df.index
    data.loc[idx, features] = scaler.fit_transform(df[features])
    scalers[sid] = scaler

# ======================
# 3. Build tensor (time × stations × features)
# ======================
dates = data['date'].unique()
num_stations = len(stations)
num_features = len(features)

tensor = np.zeros((len(dates), num_stations, num_features))

for t, d in enumerate(dates):
    df_day = data[data['date']==d]
    for _, row in df_day.iterrows():
        tensor[t, row['station_idx'], :] = row[features].values

# ======================
# 4. Build adjacency (kNN from coordinates)
# ======================
coords = stations[['latitude', 'longitude']].values
dist = distance_matrix(coords, coords)
k = 3
adj = np.zeros((num_stations, num_stations))
for i in range(len(coords)):
    knn = np.argsort(dist[i])[1:k+1]
    for j in knn:
        adj[i,j] = 1
        adj[j,i] = 1
# add self loops
np.fill_diagonal(adj, 1)

# normalize adjacency
D = np.diag(np.power(adj.sum(1), -0.5))
A_hat = D @ adj @ D
A_hat = torch.tensor(A_hat, dtype=torch.float32)

# ======================
# 5. Build sequences (30 days → next day displacement)
# ======================
seq_len = 30
X, Y = [], []

for t in range(len(dates)-seq_len-1):
    X.append(tensor[t:t+seq_len])           # (30, stations, features)
    Y.append(tensor[t+seq_len, :, 0:3])     # predict dN,dE,dU

X = torch.tensor(np.array(X), dtype=torch.float32)   # (samples, 30, stations, features)
Y = torch.tensor(np.array(Y), dtype=torch.float32)   # (samples, stations, 3)

# ======================
# 6. Train/Val/Test split
# ======================
num_samples = X.shape[0]
train_size = int(0.7 * num_samples)
val_size   = int(0.15 * num_samples)

X_train, Y_train = X[:train_size], Y[:train_size]
X_val,   Y_val   = X[train_size:train_size+val_size], Y[train_size:train_size+val_size]
X_test,  Y_test  = X[train_size+val_size:], Y[train_size+val_size:]

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ----------------------------
# Graph Attention Layer (PyTorch-only)
# ----------------------------
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, alpha=0.2):
        super(GraphAttentionLayer, self).__init__()
        self.W = nn.Parameter(torch.empty(size=(in_features, out_features)))
        nn.init.xavier_uniform_(self.W.data, gain=1.414)
        self.a = nn.Parameter(torch.empty(size=(2*out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)
        self.leakyrelu = nn.LeakyReLU(alpha)

    def forward(self, x, adj, return_attention=False):
        Wh = torch.matmul(x, self.W)
        B, N, F_ = Wh.shape
        Whi = Wh.unsqueeze(2).repeat(1, 1, N, 1)
        Whj = Wh.unsqueeze(1).repeat(1, N, 1, 1)
        e = self.leakyrelu(torch.matmul(torch.cat([Whi, Whj], dim=3), self.a).squeeze(3))
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=2)
        attention = F.dropout(attention, p=0.2, training=self.training)
        h_prime = torch.matmul(attention, Wh)
        h_prime = F.elu(h_prime)
        if return_attention:
            return h_prime, attention
        return h_prime
# ----------------------------
# GAT-LSTM Model
# ----------------------------
class GAT_LSTM(nn.Module):
    def __init__(self, in_features, gat_hidden, lstm_hidden, out_features):
        super(GAT_LSTM, self).__init__()
        self.gat = GraphAttentionLayer(in_features, gat_hidden)
        self.lstm = nn.LSTM(gat_hidden, lstm_hidden, batch_first=True)
        self.fc   = nn.Linear(lstm_hidden, out_features)

    def forward(self, x, adj):
        # x: (batch, seq_len, stations, features)
        B, T, N, F = x.shape
        seq_out = []
        for t in range(T):
            xt = x[:, t, :, :]              # (B,N,F)
            gat_out = self.gat(xt, adj)     # (B,N,gat_hidden)
            seq_out.append(gat_out)
        seq_out = torch.stack(seq_out, dim=1)           # (B,T,N,gat_hidden)

        seq_out = seq_out.permute(0,2,1,3)              # (B,N,T,gat_hidden)
        outputs = []
        for s in range(N):
            out, (h, _) = self.lstm(seq_out[:, s, :, :])
            outputs.append(self.fc(h[-1]))
        return torch.stack(outputs, dim=1)              # (B,N,out_features)

# ======================
# 8. Training loop
# ======================
model = GAT_LSTM(
    in_features = num_features,
    gat_hidden  = 32,
    lstm_hidden = 64,
    out_features= 3
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

best_val_loss = float("inf")
patience, wait = 10, 0

for epoch in range(100):
    # train
    model.train()
    optimizer.zero_grad()
    out = model(X_train, A_hat)
    loss = loss_fn(out, Y_train)
    loss.backward()
    optimizer.step()

    # validation
    model.eval()
    with torch.no_grad():
        val_out = model(X_val, A_hat)
        val_loss = loss_fn(val_out, Y_val)

    print(f"Epoch {epoch+1}: Train Loss={loss.item():.6f}, Val Loss={val_loss.item():.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        best_model = model.state_dict()
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping!")
            break

# ======================
# 9. Evaluate on test
# ======================
model.load_state_dict(best_model)
model.eval()
with torch.no_grad():
    preds = model(X_test, A_hat).numpy()
    true  = Y_test.numpy()

preds_3d = preds.copy()
true_3d  = true.copy()

# flatten
preds = preds.reshape(-1,3)
true  = true.reshape(-1,3)

rmse = np.sqrt(mean_squared_error(true, preds))
mae = mean_absolute_error(true, preds)
r2  = r2_score(true, preds)

print(f"GCN-LSTM -> RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")
# ======================
# 10. Inverse Transform Evaluation (per station)
# ======================
preds_real, true_real = [], []

for sid, idx in station_map.items():
    scaler = scalers[sid]
    
    pred_station = preds_3d[:, idx, :]   # (samples, 3)
    true_station = true_3d[:, idx, :]    # (samples, 3)

    dummy_pred = np.zeros((pred_station.shape[0], len(features)))
    dummy_true = np.zeros((true_station.shape[0], len(features)))

    dummy_pred[:, :3] = pred_station
    dummy_true[:, :3] = true_station

    inv_pred = scaler.inverse_transform(dummy_pred)[:, :3]
    inv_true = scaler.inverse_transform(dummy_true)[:, :3]

    preds_real.append(inv_pred)
    true_real.append(inv_true)

preds_real = np.vstack(preds_real)
true_real  = np.vstack(true_real)

gat_rmse = np.sqrt(mean_squared_error(true_real, preds_real))
gat_mae  = mean_absolute_error(true_real, preds_real)
gat_r2   = r2_score(true_real, preds_real)

print("\n===== Evaluation on Original Scale =====")
print(f"Real-scale RMSE = {gat_rmse:.6f}")
print(f"Real-scale MAE  = {gat_mae:.6f}")
print(f"Real-scale R²   = {gat_r2:.4f}")
preds_gat=preds_real

Preparation: data dimensions

In [ ]:
# X: (samples, 30, num_stations, num_features)
# Y: (samples, num_stations, 3)

seq_len = X.shape[1]
num_stations = X.shape[2]
num_features = X.shape[3]


X_train_np = X_train.numpy()
X_test_np  = X_test.numpy()

n_train = X_train_np.shape[0]
n_test  = X_test_np.shape[0]

X_train_flat = X_train_np.reshape(n_train, -1)
X_test_flat  = X_test_np.reshape(n_test, -1)

Build a SHAP wrapper model

In [ ]:
import torch
import torch.nn as nn


class GATLSTMWrapper(nn.Module):
    def __init__(self, base_model, adj):
        super().__init__()
        self.base_model = base_model
        self.adj = adj

    def forward(self, x):
        out = self.base_model(x, self.adj)
        dU = out[..., 2]  # (B, N)
        return dU # (B, N)

wrapped_model = GATLSTMWrapper(model, A_hat)
wrapped_model.eval()

# SHAP-based feature importance analysis
1) ‌ SHAP GAT-LSTM

Build explainer with background samples

In [ ]:
import shap

bg_size = min(50, X_train.shape[0]) # 50 background
test_size = min(50, X_test.shape[0]) # 50 

background = X_train[:bg_size] # : (bg_size, T, N, F)
X_sample = X_test[:test_size] # : (test_size, T, N, F)

DeepExplainer SHAP

In [ ]:
explainer = shap.DeepExplainer(wrapped_model, background)

shap_values_list = explainer.shap_values(X_sample, check_additivity=False)

shap_values = shap_values_list[0] # : (test_size, T, N, F)

In [ ]:
print(type(shap_values))
print(np.array(shap_values).shape)
print(len(shap_values_list))
print(np.array(shap_values_list[0]).shape)

Aggregate feature importance at feature level

In [ ]:
import numpy as np

if isinstance(shap_values_list, list):
    shap_values = np.mean(np.abs(np.array(shap_values_list)), axis=0)
else:
    shap_values = np.abs(np.array(shap_values_list))

print("SHAP values shape:", shap_values.shape)

mean_abs_shap = np.mean(shap_values, axis=(0, 1, 2))

if mean_abs_shap.ndim > 1:
    mean_abs_shap = np.mean(mean_abs_shap, axis=0)

print("Final SHAP mean shape:", mean_abs_shap.shape)

feature_importance = dict(zip(features, mean_abs_shap))
feature_importance_sorted = dict(sorted(feature_importance.items(), key=lambda x: float(x[1]), reverse=True))

print("\nFeature Importance (sorted):")
for name, val in feature_importance_sorted.items():
    print(f"{name}: {val:.6f}")

In [ ]:
import matplotlib.pyplot as plt

names = list(feature_importance_sorted.keys())
vals  = list(feature_importance_sorted.values())

plt.figure(figsize=(8,4))
plt.bar(names, vals)
plt.xticks(rotation=45)
plt.ylabel("Mean |SHAP|")
plt.title("Feature importance for dU (GAT-LSTM, Deep SHAP)")
plt.tight_layout()
plt.show()

# Permutation-based feature sensitivity analysis

In [ ]:
from sklearn.metrics import mean_squared_error

model.eval()
X_val_copy = X_val.clone()
baseline_pred = model(X_val_copy, A_hat).detach().numpy()
baseline_rmse = np.sqrt(mean_squared_error(Y_val.reshape(-1,3), baseline_pred.reshape(-1,3)))

feature_names = features
importances = []

for i, fname in enumerate(feature_names):
    X_val_permuted = X_val_copy.clone()
    # shuffle feature i
    X_val_permuted[:,:,:,i] = X_val_permuted[:,:,:,i][torch.randperm(X_val_permuted.shape[0])]
    pred = model(X_val_permuted, A_hat).detach().numpy()
    rmse = np.sqrt(mean_squared_error(Y_val.reshape(-1,3), pred.reshape(-1,3)))
    importances.append(rmse - baseline_rmse)

plt.figure(figsize=(10,5))
plt.bar(feature_names, importances)
plt.xticks(rotation=45)
plt.ylabel("Increase in RMSE after Permutation")
plt.title("Feature Importance via Permutation for GAT-LSTM")
plt.show()

In [ ]:
X_test_flat = X_test.view(X_test.shape[0], -1, X_test.shape[1])  # (108, 190, 30)

print(f"Shape of reshaped X_test: {X_test_flat.shape}")

In [ ]:
print(f"Shape of X_test: {X_test.shape}")

# Dynamic attention analysis (Dynamic Attention Analysis) 1) ‌ ‌

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    xt = X_test[:, 0, :, :]              # شکل: (B, N, F) = (108, 19, 10)
    gat_out, attention = model.gat(xt, A_hat, return_attention=True)

print("attention shape:", attention.shape)

att0 = attention[0].cpu().numpy()        # (N, N)

plt.figure(figsize=(5,4))
plt.imshow(att0, cmap="viridis")
plt.colorbar()
plt.title("Attention weights (sample 0, time step 0)")
plt.xlabel("Source station")
plt.ylabel("Target station")
plt.tight_layout()
plt.show()

Collect attention for all time steps ( )

In [ ]:
import torch.nn.functional as F

model.eval()
att_all = []

with torch.no_grad():
    B, T, N, F_feat = X_test.shape

    for t in range(T):
        xt = X_test[:, t, :, :]                  # (B, N, F_feat)
        _, att_t = model.gat(xt, A_hat, return_attention=True)  # (B, N, N)
        att_all.append(att_t.unsqueeze(1))       # (B, 1, N, N)

att_all = torch.cat(att_all, dim=1)              # (B, T, N, N)
print("att_all shape:", att_all.shape) # (108, 30, 19, 19)

Aggregate attention (mean) for dynamic analysis «» ) batch → attention

In [ ]:
att_time = att_all.mean(dim=0) # : (T, N, N)

print(att_time.shape)  # (30, 19, 19)

import matplotlib.pyplot as plt

for t in [0, 10, 20]:
    plt.figure(figsize=(5,4))
    plt.imshow(att_time[t].cpu().numpy(), cmap="viridis")
    plt.colorbar()
    plt.title(f"Mean attention over batch – time step {t}")
    plt.xlabel("Source station")
    plt.ylabel("Target station")
    plt.tight_layout()
    plt.show()

Attention time series for a station pair

In [ ]:
i = 5   # target station index
j = 2   # source station index

ts_ij = att_all[:, :, i, j].mean(dim=0).cpu().numpy() # : (T,)

plt.figure(figsize=(6,3))
plt.plot(range(T), ts_ij, marker="o")
plt.xlabel("Time step")
plt.ylabel(f"Attention weight (i={i}, j={j})")
plt.title("Dynamic attention between station j→i")
plt.grid(True)
plt.tight_layout()
plt.show()

Station-level attention importance over time

In [ ]:
att_mean = att_all.mean(dim=0)          # (T, N, N)

in_att = att_mean.sum(dim=2).cpu().numpy() # : (T, N)

plt.figure(figsize=(7,4))
for i in [0, 3, 7]: # 
    plt.plot(range(T), in_att[:, i], label=f"Station {i}")
plt.xlabel("Time step")
plt.ylabel("Sum of incoming attention")
plt.title("Dynamic station importance (incoming attention)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Weighted loss for extreme-event prediction (Extreme Event Prediction)
1) (Weighted Loss Function)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedMSELoss(nn.Module):
    """
 base_weight : ‌
 extreme_weight : ‌ dU threshold 
 threshold : dU ( target )
    """
    def __init__(self, base_weight=1.0, extreme_weight=3.0, threshold=0.8):
        super(WeightedMSELoss, self).__init__()
        self.base_weight = base_weight
        self.extreme_weight = extreme_weight
        self.threshold = threshold

    def forward(self, output, target):
        loss = (output - target) ** 2          # (B, N, 3)

        # dU = target[..., 2]  ￼
        dU = target[..., 2]                    # (B, N)

        weight_mask = torch.ones_like(dU) * self.base_weight

        weight_mask[dU > self.threshold] = self.extreme_weight  # (B, N)

        weight_mask = weight_mask.unsqueeze(-1).expand_as(loss) # (B, N, 3)

        weighted_loss = (weight_mask * loss).mean()
        return weighted_loss

Training loop with weighted loss

In [ ]:
import copy

num_epochs = 100
patience = 10 # epoch val ‌
best_val_loss = float("inf")
wait = 0

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
weighted_loss_fn = WeightedMSELoss(base_weight=1.0, extreme_weight=3.0, threshold=0.8)

best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    # --- train ---
    model.train()
    optimizer.zero_grad()

    train_pred = model(X_train, A_hat)           # (B_train, N, 3)
    train_loss = weighted_loss_fn(train_pred, Y_train)

    train_loss.backward()
    optimizer.step()

    # --- validation ---
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val, A_hat)
        val_loss = F.mse_loss(val_pred, Y_val)

    print(f"Epoch {epoch+1:03d} | Train (weighted) Loss = {train_loss.item():.6f} | Val (MSE) Loss = {val_loss.item():.6f}")

    # --- early stopping ---
    if val_loss.item() < best_val_loss - 1e-6:
        best_val_loss = val_loss.item()
        best_model_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered!")
            break

model.load_state_dict(best_model_state)

Post-training evaluation on the test set ( + )

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model.eval()
with torch.no_grad():
    test_pred = model(X_test, A_hat)   # (B_test, N, 3)

test_pred_np = test_pred.numpy()
Y_test_np    = Y_test.numpy()

rmse_all = np.sqrt(mean_squared_error(Y_test_np.reshape(-1, 3),
                                      test_pred_np.reshape(-1, 3)))
print(f"Test RMSE (all points) = {rmse_all:.6f}")

dU_true = Y_test_np[..., 2]
mask_extreme = dU_true > 0.8 # threshold

if mask_extreme.any():
    dU_pred = test_pred_np[..., 2]
    rmse_extreme = np.sqrt(mean_squared_error(dU_true[mask_extreme],
                                              dU_pred[mask_extreme]))
    print(f"Test RMSE (extreme dU) = {rmse_extreme:.6f}  on {mask_extreme.sum()} points")
else:
    print("No extreme events in test set for given threshold.")

Helper function for real-scale evaluation (mm)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_real_scale(model, X_test, Y_test, A_hat, scalers, station_map, features,
                        du_threshold_mm=5.0):
    model.eval()
    with torch.no_grad():
        preds = model(X_test, A_hat).numpy()   # (samples, stations, 3)
        true  = Y_test.numpy()                 # (samples, stations, 3)

    rmse_norm = np.sqrt(mean_squared_error(true.reshape(-1,3),
                                           preds.reshape(-1,3)))
    mae_norm  = mean_absolute_error(true.reshape(-1,3),
                                    preds.reshape(-1,3))
    r2_norm   = r2_score(true.reshape(-1,3),
                         preds.reshape(-1,3))

    preds_real_list = []
    true_real_list  = []

    for sid, idx in station_map.items():
        scaler = scalers[sid]

        pred_station = preds[:, idx, :]   # (samples, 3)
        true_station = true[:, idx, :]    # (samples, 3)

        dummy_pred = np.zeros((pred_station.shape[0], len(features)))
        dummy_true = np.zeros((true_station.shape[0], len(features)))

        dummy_pred[:, :3] = pred_station
        dummy_true[:, :3] = true_station

        inv_pred = scaler.inverse_transform(dummy_pred)[:, :3]
        inv_true = scaler.inverse_transform(dummy_true)[:, :3]

        preds_real_list.append(inv_pred)
        true_real_list.append(inv_true)

    preds_real = np.vstack(preds_real_list)   # (samples * stations, 3)
    true_real  = np.vstack(true_real_list)

    rmse_all_real = np.sqrt(mean_squared_error(true_real, preds_real))
    mae_all_real  = mean_absolute_error(true_real, preds_real)
    r2_all_real   = r2_score(true_real, preds_real)

    du_true_real = true_real[:, 2]  
    du_pred_real = preds_real[:, 2]

    thr_m = du_threshold_mm / 1000.0  

    mask_extreme = np.abs(du_true_real) > thr_m
    n_extreme = int(mask_extreme.sum())

    if n_extreme > 0:
        rmse_extreme_real = np.sqrt(mean_squared_error(du_true_real[mask_extreme],
                                                       du_pred_real[mask_extreme]))
        mae_extreme_real  = mean_absolute_error(du_true_real[mask_extreme],
                                                du_pred_real[mask_extreme])
    else:
        rmse_extreme_real = None
        mae_extreme_real  = None

    return {
        "rmse_norm": rmse_norm,
        "mae_norm": mae_norm,
        "r2_norm": r2_norm,
        "rmse_all_real": rmse_all_real,
        "mae_all_real": mae_all_real,
        "r2_all_real": r2_all_real,
        "rmse_extreme_real": rmse_extreme_real,
        "mae_extreme_real": mae_extreme_real,
        "n_extreme": n_extreme
    }

Baseline model training (standard MSE + early stopping)

In [ ]:
import copy
import torch.nn.functional as F

baseline_model = GAT_LSTM(
    in_features = num_features,
    gat_hidden  = 32,
    lstm_hidden = 64,
    out_features= 3
)

optimizer_base = torch.optim.Adam(baseline_model.parameters(), lr=0.001)
loss_fn_base   = nn.MSELoss()

num_epochs = 100
patience   = 10
best_val_loss = float("inf")
wait = 0

best_base_state = copy.deepcopy(baseline_model.state_dict())

for epoch in range(num_epochs):
    # --- train ---
    baseline_model.train()
    optimizer_base.zero_grad()

    pred_train = baseline_model(X_train, A_hat)
    loss_train = loss_fn_base(pred_train, Y_train)
    loss_train.backward()
    optimizer_base.step()

    # --- validation ---
    baseline_model.eval()
    with torch.no_grad():
        pred_val = baseline_model(X_val, A_hat)
        loss_val = loss_fn_base(pred_val, Y_val)

    print(f"[BASE] Epoch {epoch+1:03d} | Train Loss = {loss_train.item():.6f} | Val Loss = {loss_val.item():.6f}")

    if loss_val.item() < best_val_loss - 1e-6:
        best_val_loss = loss_val.item()
        best_base_state = copy.deepcopy(baseline_model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("[BASE] Early stopping!")
            break

baseline_model.load_state_dict(best_base_state)

Baseline evaluation (real scale)

In [ ]:
base_metrics = evaluate_real_scale(
    baseline_model,
    X_test, Y_test,
    A_hat,
    scalers, station_map, features,
    du_threshold_mm=5.0
)

print("\n=== BASELINE MODEL (Real scale) ===")
print(f"RMSE_all (real)     = {base_metrics['rmse_all_real']:.6f}")
print(f"MAE_all  (real)     = {base_metrics['mae_all_real']:.6f}")
print(f"R2_all   (real)     = {base_metrics['r2_all_real']:.4f}")
print(f"#Extreme points     = {base_metrics['n_extreme']}")
if base_metrics['rmse_extreme_real'] is not None:
    print(f"RMSE_extreme (real) = {base_metrics['rmse_extreme_real']:.6f}")
    print(f"MAE_extreme  (real) = {base_metrics['mae_extreme_real']:.6f}")
else:
    print("No extreme events in test set for 5mm threshold.")

In [ ]:
class AdaptiveWeightedMSELoss(nn.Module):
    def __init__(self, base_weight=1.0, scale_factor=3.0, threshold=0.8):
        super().__init__()
        self.base_weight = base_weight
        self.scale_factor = scale_factor
        self.threshold = threshold

    def forward(self, output, target):
        # output, target: (B, N, 3)
        loss = (output - target) ** 2
        dU = target[..., 2]
        weight_mask = self.base_weight + self.scale_factor * torch.relu(dU - self.threshold)
        weight_mask = weight_mask.unsqueeze(-1).expand_as(loss)
        return (weight_mask * loss).mean()

In [ ]:
du_train = Y_train.numpy()[..., 2]
thr_norm = np.quantile(du_train, 0.9) # ‌ 
print("Normalized threshold for dU:", thr_norm)

2) Weighted model training

In [ ]:
weighted_model = GAT_LSTM(
    in_features = num_features,
    gat_hidden  = 32,
    lstm_hidden = 64,
    out_features= 3
)

optimizer_w = torch.optim.Adam(weighted_model.parameters(), lr=0.001)

# loss_fn_w = WeightedMSELoss(base_weight=1.0, extreme_weight=1.5, threshold=thr_norm)

loss_fn_w = AdaptiveWeightedMSELoss(
    base_weight=1.0,
 scale_factor=3.0, # ‌ 2.0 
    threshold=thr_norm
)

num_epochs = 100
patience   = 10
best_val_loss = float("inf")
wait = 0

best_weighted_state = copy.deepcopy(weighted_model.state_dict())

for epoch in range(num_epochs):
    # --- train ---
    weighted_model.train()
    optimizer_w.zero_grad()

    pred_train = weighted_model(X_train, A_hat)
    loss_train = loss_fn_w(pred_train, Y_train)
    loss_train.backward()
    optimizer_w.step()

    weighted_model.eval()
    with torch.no_grad():
        pred_val = weighted_model(X_val, A_hat)
        loss_val = F.mse_loss(pred_val, Y_val)

    print(f"[ADAPTIVE] Epoch {epoch+1:03d} | Train (weighted) Loss = {loss_train.item():.6f} | Val (MSE) Loss = {loss_val.item():.6f}")

    if loss_val.item() < best_val_loss - 1e-6:
        best_val_loss = loss_val.item()
        best_weighted_state = copy.deepcopy(weighted_model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("[ADAPTIVE] Early stopping!")
            break

weighted_model.load_state_dict(best_weighted_state)

3) Weighted model evaluation (real scale)

In [ ]:
weighted_metrics = evaluate_real_scale(
    weighted_model,
    X_test, Y_test,
    A_hat,
    scalers, station_map, features,
    du_threshold_mm=5.0
)

print("\n=== WEIGHTED MODEL (Real scale) ===")
print(f"RMSE_all (real)     = {weighted_metrics['rmse_all_real']:.6f}")
print(f"MAE_all  (real)     = {weighted_metrics['mae_all_real']:.6f}")
print(f"R2_all   (real)     = {weighted_metrics['r2_all_real']:.4f}")
print(f"#Extreme points     = {weighted_metrics['n_extreme']}")
if weighted_metrics['rmse_extreme_real'] is not None:
    print(f"RMSE_extreme (real) = {weighted_metrics['rmse_extreme_real']:.6f}")
    print(f"MAE_extreme  (real) = {weighted_metrics['mae_extreme_real']:.6f}")
else:
    print("No extreme events in test set for 5mm threshold.")

Weighted-loss model training 1) ( dU ‌)

In [ ]:
du_train = Y_train.numpy()[..., 2]
thr_norm = np.quantile(du_train, 0.9) # top 10%

print("Normalized dU threshold (90th percentile):", thr_norm)


class WeightedMSELoss(nn.Module):
    def __init__(self, base_weight=1.0, extreme_weight=3.0, threshold=0.8):
        super(WeightedMSELoss, self).__init__()
        self.base_weight = base_weight
        self.extreme_weight = extreme_weight
        self.threshold = threshold

    def forward(self, output, target):
        # output, target: (B, N, 3)
        loss = (output - target) ** 2      # (B, N, 3)
        dU   = target[..., 2]              # (B, N)

        weight_mask = torch.ones_like(dU) * self.base_weight
        weight_mask[dU > self.threshold] = self.extreme_weight

        weight_mask = weight_mask.unsqueeze(-1).expand_as(loss)
        weighted_loss = (weight_mask * loss).mean()
        return weighted_loss

Fixed-weight loss model (Fixed Weighted)

In [ ]:
import copy
import torch.nn.functional as F

fixed_weight_model = GAT_LSTM(
    in_features = num_features,
    gat_hidden  = 32,
    lstm_hidden = 64,
    out_features= 3
)

du_train = Y_train.numpy()[..., 2]
thr_norm = np.quantile(du_train, 0.9)
print("Normalized threshold for dU:", thr_norm)

num_epochs = 100
patience = 10
best_val_loss = float("inf")
wait = 0

optimizer_fw = torch.optim.Adam(fixed_weight_model.parameters(), lr=0.001)
weighted_loss_fn = WeightedMSELoss(base_weight=1.0, extreme_weight=3.0, threshold=thr_norm)

best_fw_state = copy.deepcopy(fixed_weight_model.state_dict())

for epoch in range(num_epochs):
    fixed_weight_model.train()
    optimizer_fw.zero_grad()

    train_pred = fixed_weight_model(X_train, A_hat)
    train_loss = weighted_loss_fn(train_pred, Y_train)
    train_loss.backward()
    optimizer_fw.step()

    fixed_weight_model.eval()
    with torch.no_grad():
        val_pred = fixed_weight_model(X_val, A_hat)
        val_loss = F.mse_loss(val_pred, Y_val)

    print(f"[FIXED WEIGHTED] Epoch {epoch+1:03d} | Train (weighted) Loss = {train_loss.item():.6f} | Val (MSE) Loss = {val_loss.item():.6f}")

    if val_loss.item() < best_val_loss - 1e-6:
        best_val_loss = val_loss.item()
        best_fw_state = copy.deepcopy(fixed_weight_model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("[FIXED WEIGHTED] Early stopping!")
            break

fixed_weight_model.load_state_dict(best_fw_state)

In [ ]:
fixed_weight_metrics = evaluate_real_scale(
    fixed_weight_model,
    X_test, Y_test,
    A_hat,
    scalers, station_map, features,
    du_threshold_mm=5.0
)

print("\n=== FIXED WEIGHTED MODEL (Real scale) ===")
print(f"RMSE_all_real     = {fixed_weight_metrics['rmse_all_real']:.6f}")
print(f"RMSE_extreme_real = {fixed_weight_metrics['rmse_extreme_real']:.6f}")
print(f"R2_all_real       = {fixed_weight_metrics['r2_all_real']:.4f}")

Comparison plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = ["Baseline", "Fixed\nWeighted", "Adaptive\nWeighted"]
rmse_all = [0.006496, 0.017593, 0.018598]
rmse_ext = [0.009363, 0.011602, 0.011273]

x = np.arange(len(models))
width = 0.35

color_all = "#5DADE2" # 
color_ext = "#48C9B0" # ‌ 

fig, ax = plt.subplots(figsize=(7,5), facecolor="#F9FBFD")

bars1 = ax.bar(x - width/2, rmse_all, width, color=color_all, label="RMSE (All)")
bars2 = ax.bar(x + width/2, rmse_ext, width, color=color_ext, label="RMSE (Extreme dU>5mm)")

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel("RMSE (m)", fontsize=12)
ax.set_title("Comparison of RMSE for Baseline and Weighted GAT-LSTM Models",
             fontsize=13, pad=12, color="#2C3E50")

ax.legend(frameon=False, loc="upper left")

ax.grid(axis="y", linestyle="--", alpha=0.4, color="#AAB7B8")

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.0004,
                f"{height:.3f}", ha="center", va="bottom",
                fontsize=10, color="#1B2631")

plt.tight_layout()
plt.show()

In [ ]:
r2_all = [0.8366, 0.5869, 0.5850]
models = ["Baseline", "Fixed\nWeighted", "Adaptive\nWeighted"]
color_r2 = "#7FB3D5" # - 

fig, ax = plt.subplots(figsize=(6,4), facecolor="#F9FBFD")

bars = ax.bar(models, r2_all, color=color_r2, width=0.55, label="R² (All samples)")

ax.set_ylabel("R²", fontsize=12)
ax.set_title("R² Comparison of Baseline and Weighted GAT-LSTM Models",
             fontsize=13, pad=12, color="#2C3E50")

ax.legend(frameon=False, loc="upper right")

ax.grid(axis="y", linestyle="--", alpha=0.4, color="#AAB7B8")

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.01,
            f"{height:.2f}", ha="center", va="bottom",
            fontsize=10, color="#1B2631")

plt.tight_layout()
plt.show()

1) Short training with early stopping (per run)

In [ ]:
import copy
import torch.nn.functional as F

def train_gat_lstm_once(num_epochs=50, patience=5):
   

    model = GAT_LSTM(
        in_features = num_features,
        gat_hidden  = 32,
        lstm_hidden = 64,
        out_features= 3
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn   = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    wait = 0

    for epoch in range(num_epochs):
        # --------- train ----------
        model.train()
        optimizer.zero_grad()
        pred_train = model(X_train, A_hat)
        loss_train = loss_fn(pred_train, Y_train)
        loss_train.backward()
        optimizer.step()

        # --------- validation ----------
        model.eval()
        with torch.no_grad():
            pred_val = model(X_val, A_hat)
            loss_val = loss_fn(pred_val, Y_val)

        # print(f"[Run train] Epoch {epoch+1} | Train={loss_train.item():.4f} | Val={loss_val.item():.4f}")

        if loss_val.item() < best_val_loss - 1e-6:
            best_val_loss = loss_val.item()
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                # print("Early stopping in this run.")
                break

    model.load_state_dict(best_state)
    return model

2) Extract attention time series for one model

In [ ]:
def get_attention_series(model, X_test, A_hat, i=5, j=2):
    
    model.eval()
    B, T, N, F = X_test.shape
    att_series = []

    with torch.no_grad():
        for t in range(T):
            xt = X_test[:, t, :, :]                # (B, N, F)
            _, att_t = model.gat(xt, A_hat, return_attention=True)  # (B, N, N)
            att_mean = att_t.mean(dim=0)           # میانگین روی batch → (N, N)
            att_series.append(att_mean[i, j].item())

    return att_series

3) Multi-run averaging and plotting (mean ± std)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_runs = 5 # ‌ 3 10 
i_target = 5 # 
j_source = 2 # 

all_runs = []

for run in range(num_runs):
    print(f"=== Run {run+1}/{num_runs} ===")
    model_run = train_gat_lstm_once(num_epochs=40, patience=5)
    att_series = get_attention_series(model_run, X_test, A_hat, i=i_target, j=j_source)
    all_runs.append(att_series)

all_runs = np.array(all_runs) # : (num_runs, T)
mean_series = all_runs.mean(axis=0)
std_series  = all_runs.std(axis=0)

T = X_test.shape[1]
timesteps = np.arange(T)

color_mean = "#5DADE2" # 
color_fill = "#AED6F1" # 

plt.figure(figsize=(7,3))

plt.plot(timesteps, mean_series, marker="o", linewidth=1.8,
         color=color_mean, label=f"Mean attention (i={i_target}, j={j_source})")

plt.fill_between(timesteps,
                 mean_series - std_series,
                 mean_series + std_series,
                 color=color_fill, alpha=0.5,
                 label="±1 std")

plt.xlabel("Time step")
plt.ylabel("Attention weight")
plt.title("Mean dynamic attention between stations (averaged over runs)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper right", frameon=False)

plt.tight_layout()
plt.show()